# 03 Seasonality

Remove annual seasonality from each aligned search-sales pair with additive seasonal decomposition.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose


def find_project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

aligned_pairs_path = PROCESSED_DIR / "aligned_pairs.pkl"
if not aligned_pairs_path.exists():
    raise FileNotFoundError(f"Missing {aligned_pairs_path}. Run 02_data_processing.ipynb first.")

with open(aligned_pairs_path, "rb") as f:
    aligned_pairs = pickle.load(f)

print("Aligned categories:", sorted(aligned_pairs))

In [ ]:
cleaned_pairs = {}

for category, df in aligned_pairs.items():
    print(f"\nProcessing: {category}")

    # Need at least 2 full years for decomposition.
    if len(df) < 24:
        print(f"  Skipping - not enough data ({len(df)} months)")
        continue

    search_decomp = seasonal_decompose(
        df["search"],
        model="additive",
        period=12,
        extrapolate_trend="freq",
    )

    sales_decomp = seasonal_decompose(
        df["sales"],
        model="additive",
        period=12,
        extrapolate_trend="freq",
    )

    # Keep trend + residual to remove the seasonal component.
    search_clean = search_decomp.trend + search_decomp.resid
    sales_clean = sales_decomp.trend + sales_decomp.resid

    cleaned_pairs[category] = pd.DataFrame({
        "search_clean": search_clean,
        "sales_clean": sales_clean,
    }).dropna()

    print(f"  Clean data points: {len(cleaned_pairs[category])}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"{category} - Raw vs Seasonality-Removed", fontsize=14)

    df["search"].plot(ax=axes[0, 0], title="Search (raw)")
    cleaned_pairs[category]["search_clean"].plot(
        ax=axes[0, 1],
        title="Search (cleaned)",
        color="orange",
    )
    df["sales"].plot(ax=axes[1, 0], title="Sales (raw)")
    cleaned_pairs[category]["sales_clean"].plot(
        ax=axes[1, 1],
        title="Sales (cleaned)",
        color="orange",
    )

    plt.tight_layout()
    plot_path = PROCESSED_DIR / f"{category}_seasonality.png"
    plt.savefig(plot_path)
    print("  Saved plot:", plot_path)
    plt.show()

In [ ]:
cleaned_pairs_path = PROCESSED_DIR / "cleaned_pairs.pkl"

with open(cleaned_pairs_path, "wb") as f:
    pickle.dump(cleaned_pairs, f)

print("Saved:", cleaned_pairs_path)
print("All categories cleaned and saved.")